# A+B: testing a problem with olympdev

Install with the notebook extra: `pip install -e ".[notebook]"`.

Everything below returns a view object that renders itself, so there is no
`display(HTML(...))` and no HTML in this notebook.

In [1]:
from olympdev.notebook import Workbench

wb = Workbench.open("../example-contest", jobs=4)
wb.problems()

['aplusb']

## One solution

`run` tests a solution and returns the run view: verdict, wall time and a row
per test with the time and memory actually used.

In [2]:
aplusb = wb.problem("aplusb")
aplusb.run("main")

isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem
isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem
isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem
isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem


Test,Verdict,Time (cpu/wall),Memory
tests/sample-01.txt,OK,1.0/46 ms,728 KB
tests/sample-02.txt,OK,1.0/44 ms,620 KB
tests/manual-01.txt,OK,1.0/2.0 ms,472 KB
random-01.txt,OK,1.0/1.0 ms,764 KB
random-02.txt,OK,1.0/1.0 ms,472 KB
random-03.txt,OK,1.0/1.0 ms,472 KB


## Watching the jobs

`trace=True` shows each job card as it finishes. This uses the same driver as
a plain run - it only attaches an observer - so caching and parallelism still
apply.

In [3]:
aplusb.run("TL", trace=True)

Result,0d232d61106224c15448b8e7d40030baa0b4d1a156c800050a48943ef115d1ee


Result,cf603caffe06c8014b665deb24dfa1dd2fcf800cb4262d7c3664320c65e61b7b


Result,ccf88e75e8fd2fa297eca34592747accf02e8ccf0ad606e6680cc25fb24352a0


Result,2ce880eaec45132d8f135ec54fe09320438287e705a387a1bf7cc77f0e1c6af3


Result,33f3c3f32c758748033e2528842c2685cea1560fbfd5ea03e42454232a2abfa2


Result,64a9afdcb87156450c6177b3e1ed1a385cea6efee62c33177401bd360fe7441d


... (15 outputs left out)

Result,66fe8e406c6f2abbaafa2723e278e10d4331685a7ac8b43923a25190bb1e3727


Result,c7d5f186fbd155eb93c612aa6d7dcf8a20621f09c5fe06c9a2dcaf578d2efafd


Result,854cdb14ea878b68a61a18e31e8013332da8103d4565d831182123f9ab25745f


Test,Verdict,Time (cpu/wall),Memory
tests/sample-01.txt,OK,1.0/106 ms,472 KB
tests/sample-02.txt,OK,1.0/107 ms,468 KB
tests/manual-01.txt,TL,4.0/2100 ms,472 KB
random-01.txt,TL,3.0/2001 ms,476 KB
random-02.txt,TL,4.0/2100 ms,464 KB
random-03.txt,TL,4.0/2100 ms,472 KB


## Every solution at once

`compare` runs each solution in `info.json` and lays them out as a
test x solution matrix. One cache is shared across solutions, so the checker,
generator, validator and model solution are compiled once.

In [4]:
aplusb.compare()

Test,sol-ac.cpp main OK cpu/wall 6.0/3041 ms,sol-wa.cpp WA WA cpu/wall 6.0/333 ms,sol-tl.cpp TL TL cpu/wall 15/2498 ms
tests/sample-01.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/22 ms 472 KB,OK 1.0/121 ms 472 KB
tests/sample-02.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/24 ms 472 KB,OK 1.0/118 ms 472 KB
tests/manual-01.txt,OK 1.0/1.0 ms 728 KB,WA 1.0/23 ms 472 KB,TL 4.0/2100 ms 472 KB
random-01.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/22 ms 468 KB,TL 3.0/2100 ms 472 KB
random-02.txt,OK 1.0/1.0 ms 764 KB,WA 1.0/1.0 ms 472 KB,TL 3.0/2100 ms 472 KB
random-03.txt,OK 1.0/1.0 ms 764 KB,WA 1.0/1.0 ms 472 KB,TL 3.0/2002 ms 468 KB


## Where the numbers come from

Time and memory are read from the isolate meta of each run. For a timeout the
wall clock is shown rather than CPU time, because a solution that sleeps uses
almost no CPU.

In [5]:
report = aplusb.compare()
[(c.tag, c.verdict_text, c.wall_seconds) for c in report.columns]

[('main', 'OK', 3.104233785999895),
 ('WA', 'WA', 0.3892463729989686),
 ('TL', 'TL', 2.4834159089987224)]

## Verify prepared solutions

`verify` runs every solution in `info.json` (except `ignore`) and checks that the verdicts match the solution's tag: `main`/`OK` must be all OK, `WA` must produce a WA, and so on.

In [7]:
aplusb.verify()

Test,main main OK match cpu/wall 6.0/— ms,WA WA WA match cpu/wall 7.0/— ms,TL TL TL match cpu/wall 17/— ms
tests/sample-01.txt,OK 1.0/46 ms 728 KB,WA 1.0/25 ms 472 KB,OK 1.0/106 ms 472 KB
tests/sample-02.txt,OK 1.0/44 ms 620 KB,WA 2.0/26 ms 472 KB,OK 1.0/107 ms 468 KB
tests/manual-01.txt,OK 1.0/2.0 ms 472 KB,WA 1.0/21 ms 764 KB,TL 4.0/2100 ms 472 KB
random-01.txt,OK 1.0/1.0 ms 764 KB,WA 1.0/22 ms 472 KB,TL 3.0/2001 ms 476 KB
random-02.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/1.0 ms 472 KB,TL 4.0/2100 ms 464 KB
random-03.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/1.0 ms 472 KB,TL 4.0/2100 ms 472 KB
